# CGW and chunked SGWB explorer

Run the cells from top to bottom. The notebook uses the same population generator and injection code as the project pipeline. The chunked workflow evaluates the cumulative SGWB optimal-statistic S/N one chunk at a time, saves only important binaries, and restores the live pulsars when it finishes.

The old distance-scaling consistency generator is intentionally not used here.


In [ ]:
# 1. Configuration and project-root discovery
from pathlib import Path
import json
import sys
import gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "config.py").is_file() and (candidate / "SMBHB_pop_synth.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the repository root containing config.py and "
        "SMBHB_pop_synth.py. Start Jupyter inside this repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Population and pulsar settings. These names are used by later cells.
POPULATION_CONFIG_NAME = "test"
PULSAR_SCENARIO = "baseline"
MIN_BASELINE_YEARS = 0.0
DEBUG_PULSAR_NAMES = None

# Chunked SGWB settings. Increase these only after a small run succeeds.
SGWB_TOTAL_BINARIES = 20_000
SGWB_CHUNK_SIZE = 2_000
SGWB_RANDOM_SEED = 26072001
SGWB_SNR_LOW = 3.0
SGWB_SNR_HIGH = 3.4
SGWB_TARGET_SNR = 3.2
SGWB_N_LOUDEST_PER_CHUNK = 100
SGWB_N_NEAREST_PER_CHUNK = 25
SGWB_N_EXTREMES_PER_CHUNK = 5
RUN_CHUNKED_SGWB = False

# Optional expensive CGW analysis for a retained subset.
RUN_CGW_SNR = False
N_CGW_CANDIDATES = 100
DEBUG_CGW = False
DEBUG_DENOMINATOR_THRESHOLD = 1e-3
DEBUG_RHO_SQ_THRESHOLD = 1.0

if SGWB_TOTAL_BINARIES <= 0 or SGWB_CHUNK_SIZE <= 0:
    raise ValueError("SGWB_TOTAL_BINARIES and SGWB_CHUNK_SIZE must be positive.")
if SGWB_TOTAL_BINARIES % SGWB_CHUNK_SIZE:
    raise ValueError("SGWB_TOTAL_BINARIES must be divisible by SGWB_CHUNK_SIZE.")
if not (0 < SGWB_SNR_LOW <= SGWB_SNR_HIGH):
    raise ValueError("Set a valid SGWB S/N interval.")

SGWB_N_CHUNKS = SGWB_TOTAL_BINARIES // SGWB_CHUNK_SIZE
SGWB_OUTPUT_DIR = PROJECT_ROOT / "debug" / "sgwb_chunked_population"
SGWB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Chunks: {SGWB_N_CHUNKS} x {SGWB_CHUNK_SIZE:,} binaries")


In [ ]:
# 2. Imports and project APIs
import config
import SMBHB_pop_synth as smbhb_module
from config import generate_population
from data_loader import load_pulsars, filter_pulsars_15yr, parse_pulsar_parameters
from pta_builder import build_pta_and_params
from signal_injection import inject_population_nufft, _get_psr_radec
from CGW_SNR import compute_cgw_snr_optimal_population_fast
from enterprise.pulsar import Pulsar as EnterprisePulsar
from enterprise_extensions.frequentist import optimal_statistic as opt_stat

if POPULATION_CONFIG_NAME not in config.POPULATION_CONFIGS:
    raise KeyError(
        f"Unknown population preset {POPULATION_CONFIG_NAME!r}; "
        f"choose from {sorted(config.POPULATION_CONFIGS)}"
    )

population_config = dict(config.POPULATION_CONFIGS[POPULATION_CONFIG_NAME])
print("Population preset:", POPULATION_CONFIG_NAME)
print("Noise file:", config.NOISEFILE)


In [ ]:
# 3. Load pulsars and define the authoritative time span
psrs = load_pulsars(verbose=True, scenario=PULSAR_SCENARIO)
psrs, _, Tspan_seconds = filter_pulsars_15yr(
    psrs,
    min_baseline_years=MIN_BASELINE_YEARS,
    verbose=True,
)

if DEBUG_PULSAR_NAMES is not None:
    wanted = set(DEBUG_PULSAR_NAMES)
    psrs = [psr for psr in psrs if psr.name in wanted]

if not psrs:
    raise RuntimeError("No pulsars remain after filtering.")
if Tspan_seconds <= 0:
    raise RuntimeError(f"Invalid Tspan_seconds={Tspan_seconds!r}")

# Only TOAs are snapshotted here: libstempo residuals/toas may be methods,
# and the injection path mutates stoas, which is the state we restore.
original_stoas = {
    psr.name: np.array(psr.stoas, dtype=float, copy=True)
    for psr in psrs
}
raw_noise_params = json.loads(Path(config.NOISEFILE).read_text())
parsed_noise_params = parse_pulsar_parameters(config.NOISEFILE)

print(f"Using {len(psrs)} pulsars")
print(f"Tspan = {Tspan_seconds / (365.25 * 86400.0):.4f} yr")


In [ ]:
# 4. Pulsar inspection helpers
def _array_attribute(obj, name):
    value = getattr(obj, name)
    return value() if callable(value) else value

psr_rows = []
for psr in psrs:
    toas = np.asarray(_array_attribute(psr, "toas"), dtype=float)
    ra_rad, dec_rad = _get_psr_radec(psr)
    psr_rows.append({
        "name": psr.name,
        "n_toa": len(toas),
        "ra_rad": float(ra_rad),
        "dec_rad": float(dec_rad),
        "ra_deg": float(np.degrees(ra_rad)),
        "dec_deg": float(np.degrees(dec_rad)),
        "toa_min_yr": float(np.min(toas) / (365.25 * 86400.0)),
        "toa_max_yr": float(np.max(toas) / (365.25 * 86400.0)),
    })
psr_table = pd.DataFrame(psr_rows).sort_values("name").reset_index(drop=True)
display(psr_table)


In [ ]:
# 5. Generate one ordinary in-memory population for basic inspection
population, strain_data = generate_population(
    population_config,
    smbhb_module,
    T_obs_seconds=Tspan_seconds,
    n_binaries=min(population_config["n_binaries"], SGWB_CHUNK_SIZE),
    compute_strain=True,
    seed=SGWB_RANDOM_SEED,
)

print(f"Generated {len(population):,} binaries")
print(f"Population memory: {population.memory_gb():.4f} GB")

def population_table(pop, indices=None):
    if indices is None:
        indices = np.arange(len(pop))
    indices = np.asarray(indices, dtype=int)
    fields = ("f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc",
              "ra", "dec", "psi", "iota", "phi0", "cgw_snr")
    return pd.DataFrame({
        field: np.asarray(getattr(pop, field))[indices]
        for field in fields
    }, index=indices)

display(population_table(population, np.arange(min(10, len(population)))))


In [ ]:
# 6. Chunked SGWB helpers
SOURCE_FIELDS = (
    "f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc",
    "ra", "dec", "psi", "iota", "phi0",
)

def _extreme_indices(values, n, largest):
    values = np.asarray(values, dtype=float)
    valid = np.flatnonzero(np.isfinite(values))
    if not len(valid):
        return np.array([], dtype=int)
    n = min(int(n), len(valid))
    order = np.argsort(values[valid])
    return valid[order[-n:] if largest else order[:n]]

def select_important_indices(pop):
    proxy = np.asarray(pop.h0, dtype=float) / (2.0 * np.pi * np.asarray(pop.f, dtype=float))
    selected = set(_extreme_indices(proxy, SGWB_N_LOUDEST_PER_CHUNK, True).tolist())
    selected.update(_extreme_indices(np.asarray(pop.D_comov), SGWB_N_NEAREST_PER_CHUNK, False).tolist())
    for field in ("f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc"):
        selected.update(_extreme_indices(getattr(pop, field), SGWB_N_EXTREMES_PER_CHUNK, False).tolist())
        selected.update(_extreme_indices(getattr(pop, field), SGWB_N_EXTREMES_PER_CHUNK, True).tolist())
    return np.asarray(sorted(selected), dtype=int)

def compact_sources(pop, indices, chunk_id, seed):
    result = {
        field: np.asarray(getattr(pop, field))[indices].copy()
        for field in SOURCE_FIELDS
    }
    result.update({
        "local_index": np.asarray(indices, dtype=np.int64),
        "chunk_id": np.full(len(indices), chunk_id, dtype=np.int64),
        "chunk_seed": np.full(len(indices), seed, dtype=np.int64),
    })
    return result

def restore_pulsars():
    for psr in psrs:
        psr.stoas[:] = original_stoas[psr.name]

def sgwb_snr_from_current_stoas():
    enterprise_psrs = [
        EnterprisePulsar(psr, ephem="DE440", backend="tempo2")
        for psr in psrs
    ]
    pta, _, pta_params = build_pta_and_params(
        psrs=enterprise_psrs,
        noise_params=raw_noise_params,
        Tspan=Tspan_seconds,
        include_GW=True, include_RN=True, include_WN=True,
    )
    statistic = opt_stat.OptimalStatistic(enterprise_psrs, pta=pta, orf="hd")
    _, _, _, os_value, os_sigma = statistic.compute_os(params=pta_params)
    snr = float(os_value / os_sigma)
    del statistic, pta, enterprise_psrs
    gc.collect()
    return snr


In [ ]:
# 7. Run the chunked HPC-style SGWB workflow
def run_chunked_sgwb():
    run_dir = SGWB_OUTPUT_DIR / f"seed_{SGWB_RANDOM_SEED}_{SGWB_TOTAL_BINARIES}binaries"
    source_dir = run_dir / "important_sources"
    run_dir.mkdir(parents=True, exist_ok=True)
    source_dir.mkdir(exist_ok=True)

    restore_pulsars()
    records = []
    cumulative_edges = None
    cumulative_centres = None
    cumulative_h_squared = None

    for chunk_id in range(SGWB_N_CHUNKS):
        seed = SGWB_RANDOM_SEED + chunk_id
        pop, chunk_strain = generate_population(
            population_config, smbhb_module,
            T_obs_seconds=Tspan_seconds,
            n_binaries=SGWB_CHUNK_SIZE,
            compute_strain=True, seed=seed,
        )

        edges = np.asarray(chunk_strain["bin_edges"], dtype=float)
        centres = np.asarray(chunk_strain["bin_centres"], dtype=float)
        bin_assignment = np.asarray(chunk_strain["bin_assignment"], dtype=int)
        h_square_individual = np.asarray(
            chunk_strain["h_square_individual"], dtype=float
        )
        in_range = (bin_assignment >= 0) & (bin_assignment < len(centres))
        h_squared = np.bincount(
            bin_assignment[in_range],
            weights=h_square_individual[in_range],
            minlength=len(centres),
        )
        h_squared = h_squared[:len(centres)]

        if cumulative_h_squared is None:
            cumulative_edges = edges
            cumulative_centres = centres
            cumulative_h_squared = np.zeros_like(h_squared, dtype=float)
        elif not np.array_equal(cumulative_edges, edges):
            raise RuntimeError("Inconsistent strain-bin grids between chunks.")
        cumulative_h_squared += h_squared

        important = select_important_indices(pop)
        np.savez_compressed(
            source_dir / f"chunk_{chunk_id:04d}.npz",
            **compact_sources(pop, important, chunk_id, seed),
        )

        # inject_population_nufft adds this chunk to the current stoas; no
        # reset occurs inside the loop, so the signal remains cumulative.
        inject_population_nufft(
            psrs, pop, verbose=False, eps=1e-6,
            cache_precomputed_amplitudes=True,
        )
        snr = sgwb_snr_from_current_stoas()
        total = (chunk_id + 1) * SGWB_CHUNK_SIZE
        records.append({
            "chunk_id": chunk_id,
            "seed": seed,
            "total_generated": total,
            "important_saved": len(important),
            "cumulative_sgwb_snr": snr,
        })
        print(f"chunk {chunk_id + 1}/{SGWB_N_CHUNKS}: cumulative S/N={snr:.5f}")

        del pop, chunk_strain, important, h_squared
        gc.collect()

        if snr >= SGWB_SNR_LOW:
            break

    h_c_total = np.sqrt(
        cumulative_h_squared * cumulative_centres / np.diff(cumulative_edges)
    )
    np.savez_compressed(
        run_dir / "cumulative_background.npz",
        bin_edges_hz=cumulative_edges,
        bin_centres_hz=cumulative_centres,
        h_squared_per_bin=cumulative_h_squared,
        h_c_total=h_c_total,
    )
    manifest = {
        "Tspan_seconds": float(Tspan_seconds),
        "chunk_size": SGWB_CHUNK_SIZE,
        "chunks_processed": len(records),
        "records": records,
        "final_sgwb_snr": records[-1]["cumulative_sgwb_snr"],
        "source_directory": str(source_dir),
    }
    (run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    restore_pulsars()
    return {"run_dir": run_dir, "manifest": manifest}

chunked_sgwb_result = run_chunked_sgwb() if RUN_CHUNKED_SGWB else None
if chunked_sgwb_result is None:
    print("Skipped chunked SGWB run; set RUN_CHUNKED_SGWB=True after the setup cells succeed.")


In [ ]:
# 8. Inspect the chunked run and load only retained sources
if chunked_sgwb_result is not None:
    manifest = chunked_sgwb_result["manifest"]
    chunk_history = pd.DataFrame(manifest["records"])
    display(chunk_history)

    source_files = sorted(Path(manifest["source_directory"]).glob("chunk_*.npz"))
    retained_rows = []
    for source_file in source_files:
        data = np.load(source_file)
        n = len(data["f"])
        for i in range(n):
            row = {field: float(data[field][i]) for field in SOURCE_FIELDS}
            row.update({
                "chunk_id": int(data["chunk_id"][i]),
                "local_index": int(data["local_index"][i]),
            })
            retained_rows.append(row)
    important_source_table = pd.DataFrame(retained_rows)
    important_source_table["residual_proxy_s"] = (
        important_source_table["h0"] / (2.0 * np.pi * important_source_table["f"])
    )
    important_source_table = important_source_table.sort_values(
        "residual_proxy_s", ascending=False
    ).reset_index(drop=True)
    print(f"Retained {len(important_source_table):,} important sources.")
    display(important_source_table.head(20))
else:
    important_source_table = pd.DataFrame()


In [ ]:
# 9. Optional CGW S/N for only the retained loud candidates
if RUN_CGW_SNR and not important_source_table.empty:
    candidate_table = important_source_table.head(N_CGW_CANDIDATES).copy()
    candidate_binaries = [
        SimpleNamespace(**{field: float(row[field]) for field in SOURCE_FIELDS}, cgw_snr=0.0)
        for _, row in candidate_table.iterrows()
    ]
    enterprise_psrs = [EnterprisePulsar(psr, ephem="DE440", backend="tempo2") for psr in psrs]
    pta, _, _ = build_pta_and_params(
        psrs=enterprise_psrs, noise_params=raw_noise_params, Tspan=Tspan_seconds,
        include_GW=True, include_RN=True, include_WN=True,
    )
    candidate_snrs, candidate_breakdowns = compute_cgw_snr_optimal_population_fast(
        psrs=enterprise_psrs, pta=pta, population=candidate_binaries,
        raw_noise_params=raw_noise_params, parsed_noise_params=parsed_noise_params,
        Tspan=Tspan_seconds, profile=True, return_breakdown=True,
        debug=DEBUG_CGW, debug_denominator_threshold=DEBUG_DENOMINATOR_THRESHOLD,
        debug_rho_sq_threshold=DEBUG_RHO_SQ_THRESHOLD,
    )
    candidate_table["recomputed_cgw_snr"] = np.asarray(candidate_snrs, dtype=float)
    display(candidate_table.sort_values("recomputed_cgw_snr", ascending=False).head(20))
else:
    candidate_snrs, candidate_breakdowns = None, None
    print("Skipped retained-source CGW S/N.")


In [ ]:
# 10. Final cleanup
restore_pulsars()
print("Restored all live pulsars to their original TOAs.")
